# Act 3 — Memory and Code Execution

In [ ]:
import sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv(override=True)

from litellm import acompletion
from mini_agent import Agent, execute_python

## First, no memory
Each call to the model is completely independent — nothing carries over unless we explicitly send it back.

In [ ]:
r1 = await acompletion(model="deepseek/deepseek-chat", messages=[{"role": "user", "content": "My favorite number is 42."}])
print(r1.choices[0].message.content)

r2 = await acompletion(model="deepseek/deepseek-chat", messages=[{"role": "user", "content": "What's my favorite number?"}])
print(r2.choices[0].message.content)

It has no idea — the second call started from a blank slate, with no knowledge that the first one ever happened.

## Message history is what gives rise to memory
Keep every message — what we said, what the model replied — and send the whole list back next time:

In [ ]:
messages = [{"role": "user", "content": "My favorite number is 42."}]
r1 = await acompletion(model="deepseek/deepseek-chat", messages=messages)
messages.append(r1.choices[0].message.model_dump())
print(r1.choices[0].message.content)

messages.append({"role": "user", "content": "What's my favorite number?"})
r2 = await acompletion(model="deepseek/deepseek-chat", messages=messages)
print(r2.choices[0].message.content)

Same two questions, but this time it remembers — because we handed the whole conversation back. That accumulated `messages` list *is* memory, nothing more exotic than that.

`Agent` does exactly this: `agent.messages` is a plain list that persists across `run()` calls.

In [ ]:
agent = Agent(instructions="You are a helpful, concise assistant.")
await agent.run("My favorite number is 42.", verbose=False)
answer = await agent.run("What's my favorite number?")
print("\nFINAL:", answer)

Peek at what it's actually remembering:

In [ ]:
for m in agent.messages:
    print(m.get("role"), "->", str(m.get("content"))[:80])

## Now, a problem it can't reliably do in its head
First, without any tools:

In [ ]:
agent_no_tools = Agent(instructions="You are a helpful assistant.")
answer = await agent_no_tools.run("What is 37 to the power of 41, exactly? Give the full integer.")
print("\nFINAL:", answer)

Look closely at the digits — large exact arithmetic is exactly where LLMs are unreliable. It can *look* careful, step-by-step, even confident, and still land on the wrong number. Let's give it a way to actually compute the answer instead of reasoning its way there.

## Let's let it write and run code
`execute_python` runs code in a local subprocess and returns stdout/stderr — good enough for a demo, not a real sandbox (no isolation from the host machine). The book's version uses a real cloud sandbox (E2B); we're trading that isolation for zero setup.

In [ ]:
agent2 = Agent(tools=[execute_python], instructions="Use execute_python for anything computational.")
answer2 = await agent2.run("Write and run Python to compute 37 to the power of 41, exactly.")
print("\nFINAL:", answer2)

Same problem, but this time it's guaranteed correct — no matter how big the number gets — because the answer comes from actually running the computation instead of reasoning about it in natural language.

**Next:** what if it doesn't have the *right tool* at all? → `04_self_extending_agent.ipynb`.